# Lab 5: HuggingFace for Computer Vision

In this lab, we will explore the HuggingFace ecosystem for Computer Vision tasks. We will cover visual embeddings with CLIP, and multimodal reasoning with Qwen3.5-0.8B.

## 1. Environment Setup & Memory Management

First, we install the necessary libraries and define a utility to keep our GPU memory clean.

In [1]:
# Install relevant libraries if in Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q -U transformers accelerate bitsandbytes torch torchvision

import torch
import gc
from PIL import Image
import requests
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

def flush_memory():
    """Utility to clear GPU memory between exercises to avoid OOM."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("Memory flushed.")

## 2. HuggingFace Pipeline API

The `pipeline` API is the fastest way to use a model. Let's classify an image of a bus.

In [2]:
from transformers import pipeline

# Exercise 1: Zero-shot Image Classification
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")
url = "https://ultralytics.com/images/bus.jpg"
result = classifier(url)

for r in result:
    print(f"{r['label']}: {r['score']:.4f}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

minibus: 0.8681
trolleybus, trolley coach, trackless trolley: 0.0862
passenger car, coach, carriage: 0.0181
streetcar, tram, tramcar, trolley, trolley car: 0.0095
school bus: 0.0029


In [3]:
# clean the memory for the next models
del classifier
flush_memory()

Memory flushed.


## 2. Vision Embeddings (CLIP)

Embeddings map images into a semantic space. CLIP connects text to images.

In [4]:
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoImageProcessor


# Exercise 3: CLIP Zero-shot Classification
clip_id = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(clip_id).to("cuda" if torch.cuda.is_available() else "cpu")
clip_proc = CLIPProcessor.from_pretrained(clip_id)

image = Image.open(requests.get("https://ultralytics.com/images/bus.jpg", stream=True).raw)
labels = ["a photo of a bus", "a photo of a city scene", "a photo of a desert"]
inputs = clip_proc(text=labels, images=image, return_tensors="pt", padding=True).to(clip_model.device)

with torch.no_grad():
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)

for label, prob in zip(labels, probs[0]):
    print(f"{label}: {prob.item():.4f}")

del clip_model, clip_proc
flush_memory()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


a photo of a bus: 0.9930
a photo of a city scene: 0.0070
a photo of a desert: 0.0000
Memory flushed.


## 3. Vision Language Models (Qwen3.5-0.8B)

The light-weight Qwen3.5-0.8B is an excelente VLM for on-device reasoning and OCR.

In [5]:
from transformers import AutoProcessor, AutoModelForImageTextToText

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-0.8B")
model = AutoModelForImageTextToText.from_pretrained("Qwen/Qwen3.5-0.8B")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "https://ultralytics.com/images/bus.jpg"},
            {"type": "text", "text": "Describe what color the vehicles in this image are."},
        ],
    }
]

inputs = processor.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


The vehicles in this image are primarily **blue** and **white**.

- The large bus is predominantly **blue** with white accents, including a large white arrow-shaped logo on the side and white


In [6]:
del model, processor
flush_memory()

Memory flushed.
